## Runtime Context (`RunnableConfig`)

In the memory notebook we passed `config = {"configurable": {"thread_id": "1"}}` to the agent.
That `config` dict is LangChain's **runtime context** — a structured object that flows
through every node, tool, and runnable in the chain.

Runtime context solves a real problem: **how do you give a tool information it needs
(user ID, tenant name, feature flags) without making the LLM pass it explicitly?**

**Topics covered:**
* The `RunnableConfig` structure — built-in keys
* `tags`, `metadata`, `run_name` — for tracing and observability
* The `configurable` namespace — your custom key-value store
* `ConfigurableField` — making model parameters runtime-switchable
* Injecting config into `@tool` functions (the killer feature)
* Multi-tenant agent pattern

In [ ]:
%run langchain_common.py

## 1. The `RunnableConfig` structure

```python
config = {
    # -- Built-in LangChain/LangGraph keys --
    "run_name":        str,        # label shown in MLflow / LangSmith traces
    "tags":            list[str],  # searchable labels attached to a run
    "metadata":        dict,       # arbitrary key-value data stored with the run
    "recursion_limit": int,        # max graph recursion depth (default 25)
    "max_concurrency": int,        # parallel branch limit

    # -- Your custom namespace --
    "configurable": {
        "thread_id": "abc",        # memory: identifies the conversation thread
        # ...any other keys you define
    },
}
```

`configurable` is where you put anything application-specific.
The other keys are consumed by LangChain internals.

## 2. `tags`, `metadata`, `run_name` — observability

These keys annotate a run for filtering and searching in tracing tools like MLflow.
They cost nothing to add and pay off when debugging production issues.

In [ ]:
agent = create_agent(llm_noreason)

# Rich config — the agent behaviour is identical,
# but tracing tools can now filter/search by these values.
config = {
    "run_name":  "student-query",
    "tags":      ["cs4603", "wk3", "demo"],
    "metadata":  {"user_id": "student-42", "course": "CS4603", "env": "notebook"},
}

result = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of Pakistan?")]},
    config,
)
print(result["messages"][-1].content)

## 3. The `configurable` namespace — your custom context

You already know `thread_id` lives here. Any key-value pair you put in `configurable`
is accessible to every runnable in the chain — including tools.

Common uses:
* `user_id` — identify the current user
* `tenant_id` — identify the organisation in a SaaS app
* `feature_flags` — enable/disable behaviours per user
* `user_tier` — adjust responses for free vs. premium users

In [ ]:
# Two different sessions — same agent, different context
session_a = {"configurable": {"thread_id": "user-ali",   "user_name": "Ali"}}
session_b = {"configurable": {"thread_id": "user-sara",  "user_name": "Sara"}}

from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(llm_noreason, checkpointer=InMemorySaver())

mem_agent.invoke({"messages": [HumanMessage(content="My favourite city is Lahore.")]}, session_a)
mem_agent.invoke({"messages": [HumanMessage(content="My favourite city is Istanbul.")]}, session_b)

r_a = mem_agent.invoke({"messages": [HumanMessage(content="What is my favourite city?")]}, session_a)
r_b = mem_agent.invoke({"messages": [HumanMessage(content="What is my favourite city?")]}, session_b)

print(f"Ali's session  : {r_a['messages'][-1].content}")
print(f"Sara's session : {r_b['messages'][-1].content}")

## 4. `ConfigurableField` — runtime-switchable model parameters

`ConfigurableField` lets you declare that a chain parameter (e.g., model temperature)
can be overridden at runtime via `configurable` — without rebuilding the chain.

This is useful when:
* You want a **creative mode** (high temperature) and a **precise mode** (low temperature)
* Different user tiers get different LLM models
* You want to A/B test prompts or models without redeploying

In [ ]:
from langchain_core.runnables import ConfigurableField

# Declare temperature as a configurable field with a default
configurable_llm = llm_noreason.configurable_fields(
    temperature=ConfigurableField(
        id="llm_temperature",
        name="LLM Temperature",
        description="0 = deterministic / precise, 1 = creative / random",
    )
)

prompt = "Write a one-sentence tagline for a coffee shop."

# Default temperature (0 — deterministic)
result_precise = configurable_llm.invoke(prompt)
print("Precise (temp=0):  ", result_precise.content)

# High temperature at runtime — no code rebuild needed
result_creative = configurable_llm.invoke(
    prompt,
    config={"configurable": {"llm_temperature": 1.0}},
)
print("Creative (temp=1): ", result_creative.content)

## 5. Injecting config into `@tool` functions ⭐

This is the most powerful runtime-context feature.

Add `config: RunnableConfig` as a parameter to any `@tool` function.
LangChain **injects the live config automatically** at call time — the LLM never sees this
parameter and does not need to provide it.

This lets tools access `user_id`, `tenant_id`, or any other context that was
set when `agent.invoke(...)` was called, without the user mentioning it in their message.

In [ ]:
from langchain_core.runnables import RunnableConfig

# Simulated user account database
USER_ACCOUNTS = {
    "user-001": {"name": "Ali",  "balance": 5000.00, "tier": "standard"},
    "user-002": {"name": "Sara", "balance": 25000.00, "tier": "premium"},
}

@tool
def get_account_balance(config: RunnableConfig) -> str:
    """
    Get the current account balance for the logged-in user.
    No user ID is required — it is read from the session context automatically.
    """
    # Extract user_id from the injected config — the LLM never supplies this
    user_id = config.get("configurable", {}).get("user_id")

    if not user_id:
        return "Error: no user session found. Please log in."

    account = USER_ACCOUNTS.get(user_id)
    if not account:
        return f"Error: account not found for user_id '{user_id}'."

    return f"Hello {account['name']}! Your balance is ${account['balance']:,.2f} (tier: {account['tier']})."


banking_agent = create_agent(llm_noreason, [get_account_balance])

# Same question — different answer depending on who is logged in
for uid in ["user-001", "user-002", "user-999"]:
    ctx = {"configurable": {"user_id": uid}}
    result = banking_agent.invoke({"messages": [HumanMessage(content="What is my balance?")]}, ctx)
    print(f"[user_id={uid}] {result['messages'][-1].content}")
    print()

## 6. Multi-tenant pattern — full example

Combine config injection, memory, and per-tenant context into a realistic pattern:
one agent instance serves many users, each isolated by `user_id` + `thread_id`.

In [ ]:
from pydantic import BaseModel, Field

ORDERS = {
    "user-001": [{"id": "ORD-11", "item": "Laptop",  "status": "Shipped"},
                 {"id": "ORD-12", "item": "Mouse",   "status": "Delivered"}],
    "user-002": [{"id": "ORD-21", "item": "Monitor", "status": "Processing"}],
}

@tool
def list_my_orders(config: RunnableConfig) -> str:
    """List all orders belonging to the currently logged-in user."""
    user_id = config.get("configurable", {}).get("user_id", "")
    orders = ORDERS.get(user_id, [])
    if not orders:
        return "No orders found for your account."
    lines = [f"  {o['id']}: {o['item']} — {o['status']}" for o in orders]
    return "Your orders:\n" + "\n".join(lines)


@tool
def get_account_info(config: RunnableConfig) -> str:
    """Return the profile information for the currently logged-in user."""
    user_id = config.get("configurable", {}).get("user_id", "")
    account = USER_ACCOUNTS.get(user_id)
    if not account:
        return "Account not found."
    return f"Name: {account['name']}, Tier: {account['tier']}, Balance: ${account['balance']:,.2f}"


multi_agent = create_agent(
    llm_noreason,
    [list_my_orders, get_account_info],
    checkpointer=InMemorySaver(),
)

def build_config(user_id: str, session_id: str) -> dict:
    return {"configurable": {"user_id": user_id, "thread_id": session_id}}


# Ali's session
ali_cfg   = build_config("user-001", "ali-session-1")
# Sara's session
sara_cfg  = build_config("user-002", "sara-session-1")

questions = [
    (ali_cfg,  "Show me my orders."),
    (sara_cfg, "What is my account info?"),
    (ali_cfg,  "What is my account balance?"),
]

for cfg, q in questions:
    result = multi_agent.invoke({"messages": [HumanMessage(content=q)]}, cfg)
    uid = cfg["configurable"]["user_id"]
    print(f"[{uid}] Q: {q}")
    print(f"        A: {result['messages'][-1].content}")
    print()

## 7. `with_config` — setting default config on a runnable

`chain.with_config(config)` returns a new runnable with the given config baked in as defaults.
Callers can still override any key. Useful for creating environment-specific variants
(dev / staging / prod) without passing config manually on every call.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_messages([("human", "{question}")])
    | llm_noreason
    | StrOutputParser()
)

# Bind default metadata for all runs from this variant
prod_chain = chain.with_config(
    run_name="prod-query",
    tags=["production", "cs4603"],
    metadata={"env": "prod"},
)

dev_chain = chain.with_config(
    run_name="dev-query",
    tags=["development", "cs4603"],
    metadata={"env": "dev"},
)

# Both use the same chain logic; tracing tools will distinguish them by tag/metadata
print(prod_chain.invoke({"question": "What is 2+2?"}))
print(dev_chain.invoke({"question": "What is 2+2?"}))

## Summary

| Feature | Key API | Use case |
|---|---|---|
| Observability labels | `tags`, `metadata`, `run_name` | Filter traces in MLflow / LangSmith |
| Custom runtime data | `configurable: {...}` | Pass user_id, tenant_id, feature flags |
| Runtime param overrides | `ConfigurableField` | Switch temperature / model per call |
| Config injection in tools | `config: RunnableConfig` param | Multi-tenant data access without LLM involvement |
| Bound defaults | `chain.with_config(...)` | Environment-specific chain variants |

The key insight: **the `config` dict flows invisibly through the entire chain**.
You set it once at the top-level `invoke` call, and every tool and runnable downstream
can read from it — no need to thread extra arguments through every function.